In [29]:
import pandas  as pd
import os
from dotenv import load_dotenv, find_dotenv
import pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

# Data Loading and Preprocessing

In [9]:
files = pd.read_csv("course_descriptions.csv", encoding="cp1252")

In [11]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is  {row["course_slug"]}, 
    the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [13]:
pd.set_option("display.max_rows", 106)
files['course_description_new'] = files.apply(create_course_description, axis=1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is  sql, \n  ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is  st...
14     The course name is Mathematics, the slug is  m...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is  p...
17     The course name is Start

In [18]:
%load_ext dotenv
%dotenv
%reload_ext dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [19]:
load_dotenv(find_dotenv(), override=True)

True

In [20]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [27]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [28]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index not in index list.


In [26]:
pc.list_indexes()

[
    {
        "name": "text",
        "metric": "cosine",
        "host": "text-z3funzb.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    }
]

In [30]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-z3funzb.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [31]:
index = pc.Index(index_name)

# Embedding the data

In [32]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [39]:
def create_embeddings(row):
    combined_text = " ".join([str(row[field]) for field in ['course_description','course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar=False)
    return embedding

In [40]:
files['embedding'] = files.apply(create_embeddings, axis=1)

In [36]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors= vectors_to_upsert)

{'upserted_count': 106}

# Semantic serach

In [44]:
query = "clustering"
query_embedding = model.encode(query, show_progress_bar= False).tolist()

In [45]:
query_results = index.query(
    vector= [query_embedding],
    top_k=12,
    include_values=True
)

In [46]:
query_results

{'matches': [{'id': 'Machine Learning in Excel',
              'score': 0.354952842,
              'values': [-0.0183002371,
                         -0.02794859,
                         -0.0253203604,
                         -0.0126938857,
                         -0.0240366571,
                         -0.0219841078,
                         -0.0511236973,
                         -0.0535799898,
                         0.00997657236,
                         0.0282286219,
                         -0.040832486,
                         -0.0362686925,
                         0.0683277547,
                         -0.0348471254,
                         -0.00728512369,
                         0.0366662964,
                         -0.003310167,
                         -0.00411816873,
                         -4.75542038e-05,
                         -0.0627968833,
                         0.0846960247,
                         0.0300104804,
                         -0.0528304875,
